# Explore beam transport with ImpactX

A **100 MeV electron bunch** enters HTU: quadrupole magnets focus it, a chicane bends its path, and more magnets guide it toward an undulator. Follow how the beam shape changes along the beamline.

We track a **25 pC Gaussian bunch** with the upstream ImpactX HTU lattice. This source is independent of the WarpX run. Each macroparticle represents many electrons; increasing their number samples the same physical bunch more finely.

**Use the WarpX CPU kernel (which also includes ImpactX)** and run from this notebook's `htu` folder. Start with the three settings below. The shared [helper script](tutorial_helpers.py) handles the HTU lattice, source sampling, execution, and plots.

🔬 **Model boundaries:** prescribed magnets, no space charge, no radiation, and no modeled apertures. The screens record the simulated beam without clipping it. With no collective fields, particles track independently—a useful contrast to WarpX's PIC workload.


In [ ]:
from tutorial_helpers import beam_explorer, compare_cost, plot_beam_sizes, run_beams

particles = 10_000
total_energies_MeV = [100]  # Total energy, including rest energy.
chicane_r56_um = 200.0  # Upstream chicane control setting.

## 1 · Run the simulation

Predict the beam shape after the first focusing magnets. Then run the simulation.

Each call creates a fresh folder under `runs/`, with diagnostics, a text log for each energy, and `performance.json`. The timer includes process startup, tracking, and writing diagnostics, but excludes plotting. We request **two CPU threads** for every run.


In [ ]:
run = run_beams(particles, total_energies_MeV, chicane_r56_um)

## 2 · Explore the beam screens 🔍

Press **Play screens** or drag the slider. These are saved transverse snapshots, not a movie in physical time. Turn off **Zoom to fit** to compare sizes on fixed axes and see how much the beam expands from the source.

Find the first focusing screen (`TCPhosphor`), the middle of the chicane (`ChicaneSlit`), and the exit. Does the beam stay round? Read the axes: zoomed maps can switch between µm and mm. Colors show charge per bin, not density per unit area.

The viewer also saves `beam_explorer.html` (open it in a browser if notebook HTML is blocked) in the run folder.


In [ ]:
beam_explorer(run)

## 3 · Follow the beam sizes

The **rms size** measures how widely particle positions spread around their mean. Follow x and y separately: quadrupoles can focus one plane while defocusing the other.

Where is each plane smallest? Can you connect a narrow part of the curves to a screen in the viewer? The charge check below is not an aperture test: real openings are not modeled. Invalid tracking is flagged, never interpreted as physical beam loss.

The **lattice strip** below the curves shows where the quadrupoles, bending magnets, steerers, and screens sit on the same distance axis. Gaps are drifts. Match a change in beam size to the nearby elements. Element lengths are to scale; symbol heights are not beam sizes or apertures. Magnets are shown even when their field is zero.


In [ ]:
plot_beam_sizes(run);

## 4 · More particles, more computing work 💻

**Predict first:** will runtime, file size, and visible beam detail all change by the same factor?

Run the next cell for a second, larger simulation with the same energy, magnets, and thread setting. The physical charge stays **25 pC**, so each macroparticle now carries less weight. Compare the two cost charts, then inspect the new beam maps and rms sizes.

Startup and file operations can dominate short runs. This compares workload sizes on fixed hardware; it is **not a parallel scaling benchmark**. More samples also do not add missing physics.


In [ ]:
larger = run_beams(10 * particles, total_energies_MeV, chicane_r56_um)
compare_cost(run, larger);

In [ ]:
beam_explorer(larger)

In [ ]:
plot_beam_sizes(larger);

## Optional · Change the beam energy

Return to the settings cell, keep `particles = 10_000`, set `total_energies_MeV = [100, 150]` and `chicane_r56_um = 0.0`, and rerun **steps 1–3 only**. Select each energy in the viewer. Which plane responds most strongly?

Both beams now use zero chicane field, following the upstream off-energy example. The magnets and the 100 MeV reference stay fixed. Shifting the mean energy preserves sampled positions and absolute momentum spreads, so the 150 MeV beam starts with smaller angular divergence and relative energy spread. The shared helper script records the upstream revision and source conventions.

Passing a WarpX bunch into ImpactX requires a consistent conversion of coordinates and units; this tutorial keeps the two examples independent.
